<a href="https://colab.research.google.com/github/keerthana-25/distracted-driver-detection/blob/main/notebooks/train_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Distracted Driver Detection — Colab Training Notebook

**Run this notebook on Google Colab with a free T4 GPU**

---

### What this notebook does
1. Sets up the environment (GPU check, dependencies)
2. Clones the project from GitHub
3. Loads the State Farm dataset (from Google Drive or Kaggle)
4. Trains EfficientNet-B3 with full MLflow tracking
5. Evaluates on test set — confusion matrix, per-class accuracy
6. Generates Grad-CAM visualizations
7. Downloads the trained model to your laptop

**Estimated time:** ~45 minutes on T4 GPU (vs ~15 hours on CPU)

### Before you start
- Runtime → Change runtime type → **T4 GPU**
- Have your State Farm dataset zip ready (in Google Drive or Kaggle account)

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('❌ No GPU found!')
    print('Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Clone Project

In [ ]:
import os

REPO_URL = 'https://github.com/keerthana-25/distracted-driver-detection'
PROJECT_DIR = '/content/distracted-driver-detection'

if os.path.exists(PROJECT_DIR):
    print('Project already cloned. Pulling latest...')
    !cd {PROJECT_DIR} && git pull
else:
    print('Cloning project...')
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
print(f'Working directory: {os.getcwd()}')

## Step 3 — Install Dependencies

In [ ]:
print('Installing dependencies...')
!pip install -q \
    timm>=0.9.0 \
    torchmetrics>=1.0.0 \
    mlflow>=2.5.0 \
    gradio>=4.0.0 \
    opencv-python-headless>=4.8.0 \
    flask flask-cors \
    pyyaml tqdm \
    huggingface_hub \
    python-dotenv \
    tensorboard

# Add project to Python path
import sys
sys.path.insert(0, '/content/distracted-driver-detection')

# Verify
import timm, torchmetrics, mlflow
print(f'✅ timm {timm.__version__}')
print(f'✅ torchmetrics {torchmetrics.__version__}')
print(f'✅ mlflow {mlflow.__version__}')
print('All dependencies ready!')

## Step 4 — Load Dataset

**Choose ONE of the options below** based on how you have the data.

### Option A — From Google Drive (recommended)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('data/raw', exist_ok=True)

# Copy zip from Drive to Colab (fast — stays within Google)
# Change path below if your zip is in a subfolder
DRIVE_ZIP_PATH = '/content/drive/MyDrive/state-farm-distracted-driver-detection.zip'

if os.path.exists(DRIVE_ZIP_PATH):
    print('Copying zip from Drive...')
    !cp "{DRIVE_ZIP_PATH}" data/raw/
    print('Extracting...')
    !unzip -q data/raw/state-farm-distracted-driver-detection.zip -d data/raw/
    !ls data/raw/imgs/train/
    print('✅ Dataset ready!')
else:
    print(f'❌ Zip not found at {DRIVE_ZIP_PATH}')
    print('Upload the zip to Google Drive first, then update DRIVE_ZIP_PATH')

### Option B — From Kaggle API

In [ ]:
# Only run this cell if using Kaggle
# First accept competition rules at kaggle.com/c/state-farm-distracted-driver-detection

import os, json

KAGGLE_USERNAME = 'your_kaggle_username'   # ← change this
KAGGLE_KEY      = 'your_kaggle_api_key'    # ← change this

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
os.makedirs('data/raw', exist_ok=True)
!kaggle competitions download -c state-farm-distracted-driver-detection -p data/raw
!unzip -q data/raw/state-farm-distracted-driver-detection.zip -d data/raw
!ls data/raw/imgs/train/
print('✅ Dataset downloaded from Kaggle!')

## Step 5 — Prepare Dataset Splits

In [ ]:
import sys
sys.path.insert(0, '/content/distracted-driver-detection')

from src.data.dataset import split_dataset, compute_dataset_statistics
from pathlib import Path

# Check if already split
if Path('data/processed/train').exists():
    print('Splits already exist. Skipping...')
else:
    print('Splitting dataset 70/15/15 (stratified)...')
    split_dataset(
        source_dir='data/raw/imgs/train',
        output_dir='data/processed',
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15,
    )

# Show statistics
stats = compute_dataset_statistics('data/processed')
print('\nDataset splits:')
for split, info in stats.items():
    print(f'  {split.upper():8s}: {info["total"]:,} images')

## Step 6 — Train the Model

In [ ]:
# ── Training configuration ──
# Adjust these if needed
ARCHITECTURE = 'efficientnet_b3'   # primary model
EPOCHS       = 30                  # ~45 min on T4
BATCH_SIZE   = 32
LEARNING_RATE = 0.001
DATA_DIR     = 'data/processed'

print(f'Training config:')
print(f'  Architecture : {ARCHITECTURE}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Batch size   : {BATCH_SIZE}')
print(f'  Data dir     : {DATA_DIR}')
print(f'  Device       : {"GPU" if torch.cuda.is_available() else "CPU"}')
print()

In [ ]:
import sys
sys.path.insert(0, '/content/distracted-driver-detection')

from src.training.trainer import TrainingConfig, Trainer
from src.data.dataset import create_dataloaders

# Build config
config = TrainingConfig({
    'architecture'         : ARCHITECTURE,
    'epochs'               : EPOCHS,
    'batch_size'           : BATCH_SIZE,
    'learning_rate'        : LEARNING_RATE,
    'freeze_backbone_epochs': 3,
    'early_stopping_patience': 7,
    'num_workers'          : 2,
    'mixed_precision'      : True,
    'pretrained'           : True,
    'output_dir'           : 'models',
    'experiment_name'      : 'distracted-driver-detection',
    'run_name'             : f'colab_{ARCHITECTURE}_ep{EPOCHS}',
    'data_dir'             : DATA_DIR,
})

# Create dataloaders
dataloaders = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    image_size=224,
    num_workers=2,
    use_weighted_sampler=True,
)

print(f'Train batches : {len(dataloaders["train"])}')
print(f'Val batches   : {len(dataloaders["val"])}')
print('\nStarting training...')
print('=' * 60)

# Train
trainer = Trainer(config)
best_metrics = trainer.train(dataloaders)

print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print(f'Best Val Accuracy : {best_metrics.get("accuracy_top1", 0):.4f}')
print(f'Best Val F1       : {best_metrics.get("f1_macro", 0):.4f}')
print(f'Best Val AUROC    : {best_metrics.get("auroc", 0):.4f}')
print(f'Best Epoch        : {best_metrics.get("epoch", 0)}')

## Step 7 — Plot Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
from src.training.visualizations import plot_training_curves
import matplotlib.pyplot as plt
from pathlib import Path

history_path = 'models/training_history.json'
if Path(history_path).exists():
    fig = plot_training_curves(history_path, save_dir='docs/figures')
    plt.show()
    print('Training curves saved to docs/figures/training_curves.png')
else:
    print('No training history found')

## Step 8 — Evaluate on Test Set

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from src.model.architecture import create_model, load_checkpoint, ModelMetrics, IDX_TO_NAME
from src.training.visualizations import plot_confusion_matrix, plot_per_class_accuracy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best model
model = create_model(ARCHITECTURE, pretrained=False, device=device)
model = load_checkpoint(model, 'models/best_model.pth', device)
model.eval()

# Evaluate
metrics = ModelMetrics(num_classes=10, device=device)
loss_fn = nn.CrossEntropyLoss()
total_loss = 0.0

print('Evaluating on test set...')
with torch.no_grad():
    for images, targets in dataloaders['test']:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        total_loss += loss_fn(logits, targets).item()
        metrics.update(logits, targets)

results = metrics.compute()
results['test_loss'] = total_loss / len(dataloaders['test'])

print('\n' + '=' * 55)
print('TEST SET RESULTS')
print('=' * 55)
print(f'  Top-1 Accuracy  : {results["accuracy_top1"]:.4f} ({results["accuracy_top1"]*100:.2f}%)')
print(f'  Top-3 Accuracy  : {results["accuracy_top3"]:.4f} ({results["accuracy_top3"]*100:.2f}%)')
print(f'  F1 Macro        : {results["f1_macro"]:.4f}')
print(f'  Precision Macro : {results["precision_macro"]:.4f}')
print(f'  Recall Macro    : {results["recall_macro"]:.4f}')
print(f'  AUROC           : {results["auroc"]:.4f}')
print(f'  Test Loss       : {results["test_loss"]:.4f}')
print('\n  Per-Class Accuracy:')
for i, acc in enumerate(results['per_class_accuracy']):
    bar = '█' * int(acc * 20)
    print(f'    {IDX_TO_NAME.get(i, str(i)):25s} {acc:.3f}  {bar}')
print('=' * 55)

In [ ]:
# Plot confusion matrix and per-class accuracy
cm = np.array(results['confusion_matrix'])
plot_confusion_matrix(cm, save_dir='docs/figures')
plot_per_class_accuracy(results['per_class_accuracy'], save_dir='docs/figures')
plt.show()
print('Figures saved to docs/figures/')

## Step 9 — Grad-CAM Visualizations

In [ ]:
from src.explainability.gradcam import ExplainablePredictor
from src.data.dataset import IDX_TO_NAME
from PIL import Image
import cv2

predictor = ExplainablePredictor(model, device=device)

# Run Grad-CAM on 6 test samples
test_dataset = dataloaders['test'].dataset
n_samples = 6

fig, axes = plt.subplots(n_samples, 3, figsize=(13, 4 * n_samples))
fig.suptitle('Grad-CAM Explainability — Test Samples', fontsize=14, fontweight='bold')

for i in range(n_samples):
    path = test_dataset.samples[i * (len(test_dataset) // n_samples)]
    true_label = test_dataset.labels[i * (len(test_dataset) // n_samples)]

    pil_img = Image.open(path).convert('RGB')
    result = predictor.predict(pil_img, generate_cam=True)

    axes[i][0].imshow(pil_img.resize((224,224)))
    axes[i][0].set_title(f'True: {IDX_TO_NAME.get(true_label, str(true_label))[:22]}', fontsize=9)
    axes[i][0].axis('off')

    cam_resized = cv2.resize(result['cam'], (224, 224))
    axes[i][1].imshow(cam_resized, cmap='jet')
    axes[i][1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[i][1].axis('off')

    axes[i][2].imshow(result['cam_overlay'])
    correct = '✓' if result['predicted_class'] == true_label else '✗'
    axes[i][2].set_title(f'{correct} {result["predicted_label"][:22]}\nConf: {result["confidence"]:.1%}', fontsize=9)
    axes[i][2].axis('off')

plt.tight_layout()
plt.savefig('docs/figures/gradcam_examples.png', bbox_inches='tight', dpi=130)
plt.show()
print('Grad-CAM examples saved to docs/figures/gradcam_examples.png')

## Step 10 — Download Everything

In [ ]:
from google.colab import files

# Download trained model (most important)
print('Downloading best_model.pth...')
files.download('models/best_model.pth')

# Download training history (for plots)
print('Downloading training_history.json...')
files.download('models/training_history.json')

In [ ]:
# Download all generated figures
import os
!zip -r figures.zip docs/figures/
files.download('figures.zip')
print('Figures downloaded!')

In [ ]:
# Download MLflow experiment data
!zip -r mlruns.zip mlruns/
files.download('mlruns.zip')
print('MLflow runs downloaded!')

## Step 11 — Upload Model to HuggingFace Space

In [ ]:
# Optional — upload model directly to HuggingFace Space from Colab
# This saves you from downloading and re-uploading manually

HF_TOKEN   = 'hf_your_token_here'          # ← paste your HF token
HF_REPO_ID = 'keerthana-m/distracted-driver-detection'

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)

print('Uploading model to HuggingFace Space...')
api.upload_file(
    path_or_fileobj='models/best_model.pth',
    path_in_repo='models/best_model.pth',
    repo_id=HF_REPO_ID,
    repo_type='space',
)
print(f'✅ Model uploaded to https://huggingface.co/spaces/{HF_REPO_ID}')

---

## After training is done

**On your laptop:**

```bash
# Place downloaded files
mv ~/Downloads/best_model.pth distracted-driver-detection/models/
mv ~/Downloads/training_history.json distracted-driver-detection/models/

# Extract figures
unzip figures.zip -d distracted-driver-detection/

# Extract MLflow runs
unzip mlruns.zip -d distracted-driver-detection/

# Launch demo locally
cd distracted-driver-detection
python3 webapp/gradio_app.py

# View MLflow dashboard
mlflow ui --backend-store-uri mlruns
```